# Syllable Enrichment Between Species

Plot-only notebook for testing which MoSeq syllables are enriched or depleted by species, using the saved matched-arm soft counts from the multispecies slow-mode workflow.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, norm
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

REPO_ROOT = Path('/Users/meganbishop/slowmodeevo').resolve()
RUN_ROOT = REPO_ROOT / 'outputs/multispecies_slow_modes/global_clustering_modes/global_outputs/XY_EvenSampled_SlowModes'
SYLLABLE_PROBS_CSV = RUN_ROOT / 'matched_arm_syllable_probs.csv'
OUT_DIR = RUN_ROOT / 'syllable_enrichment_between_species'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_SYLLABLES = 40
MIN_SOFT_FRAMES = 100.0
EFFECTIVE_COUNT_SCALE = 1.0
EPS = 1e-9

if not SYLLABLE_PROBS_CSV.exists():
    raise FileNotFoundError(f'Missing {SYLLABLE_PROBS_CSV}. Run section 6f in 00_multispecies_analysis.ipynb once.')

syllable_probs = pd.read_csv(SYLLABLE_PROBS_CSV)
required = {'species', 'aligned_arm', 'moseq_cluster', 'soft_frames'}
missing = required - set(syllable_probs.columns)
if missing:
    raise ValueError(f'{SYLLABLE_PROBS_CSV} is missing required columns: {sorted(missing)}')

syllable_probs['species'] = syllable_probs['species'].astype(str)
syllable_probs['aligned_arm'] = syllable_probs['aligned_arm'].astype(int)
syllable_probs['moseq_cluster'] = syllable_probs['moseq_cluster'].astype(int)
if 'moseq_label' not in syllable_probs.columns:
    syllable_probs['moseq_label'] = syllable_probs['moseq_cluster'].astype(str)

species_order = sorted(syllable_probs['species'].unique())
arm_order = sorted(syllable_probs['aligned_arm'].unique())
print(f'Loaded {len(syllable_probs):,} rows for {len(species_order)} species, {len(arm_order)} aligned arms, {syllable_probs["moseq_cluster"].nunique()} syllables')
display(syllable_probs.head())

## Enrichment Model

Within each aligned arm, this notebook treats the soft frame table as a species-by-syllable contingency matrix. It reports log2 observed/expected enrichment under species-syllable independence, standardized residuals, and Benjamini-Hochberg adjusted residual p-values.

In [ ]:
def wrapped_syllable_label(syllable_id, width=14):
    label = str(
        syllable_probs.loc[syllable_probs['moseq_cluster'] == int(syllable_id), 'moseq_label']
        .dropna()
        .astype(str)
        .iloc[0]
        if np.any(syllable_probs['moseq_cluster'] == int(syllable_id))
        else syllable_id
    ).replace('_', ' ')
    if label.startswith(f'{int(syllable_id)}:'):
        label = label.split(':', 1)[1].strip()
    words = label.split()
    lines = []
    line = ''
    for word in words:
        candidate = f'{line} {word}'.strip()
        if len(candidate) > width and line:
            lines.append(line)
            line = word
        else:
            line = candidate
    if line:
        lines.append(line)
    if not lines:
        lines = [str(syllable_id)]
    return f'{int(syllable_id)}:\n' + '\n'.join(lines[:3])


def build_arm_enrichment_table(df, arm, effective_count_scale=EFFECTIVE_COUNT_SCALE):
    arm_df = df.loc[df['aligned_arm'] == int(arm)].copy()
    observed = (
        arm_df.pivot_table(index='species', columns='moseq_cluster', values='soft_frames', aggfunc='sum', fill_value=0.0)
        .reindex(index=species_order, fill_value=0.0)
        .sort_index(axis=1)
    )
    observed = observed.loc[:, observed.sum(axis=0) >= MIN_SOFT_FRAMES]
    scaled = observed * float(effective_count_scale)
    chi2, omnibus_p, dof, expected = chi2_contingency(scaled.to_numpy(float), correction=False)
    expected = pd.DataFrame(expected, index=observed.index, columns=observed.columns) / max(float(effective_count_scale), EPS)
    residual = (observed - expected) / np.sqrt(expected + EPS)
    log2_enrichment = np.log2((observed + EPS) / (expected + EPS))
    p_values = 2 * norm.sf(np.abs(residual.to_numpy(float)))
    q_values = np.full_like(p_values, np.nan, dtype=float)
    finite = np.isfinite(p_values)
    q_values[finite] = multipletests(p_values[finite], method='fdr_bh')[1]
    q_values = pd.DataFrame(q_values, index=observed.index, columns=observed.columns)

    long = log2_enrichment.stack().rename('log2_enrichment_vs_arm_expected').reset_index()
    long = long.merge(residual.stack().rename('standardized_residual').reset_index(), on=['species', 'moseq_cluster'])
    long = long.merge(q_values.stack().rename('q_value').reset_index(), on=['species', 'moseq_cluster'])
    long = long.merge(observed.stack().rename('soft_frames').reset_index(), on=['species', 'moseq_cluster'])
    long = long.merge(expected.stack().rename('expected_soft_frames').reset_index(), on=['species', 'moseq_cluster'])
    label_map = syllable_probs.drop_duplicates('moseq_cluster').set_index('moseq_cluster')['moseq_label']
    long['moseq_label'] = long['moseq_cluster'].map(label_map).fillna(long['moseq_cluster'].astype(str))
    long['aligned_arm'] = int(arm)
    long['omnibus_chi2'] = float(chi2)
    long['omnibus_p_value'] = float(omnibus_p)
    long['omnibus_dof'] = int(dof)
    return observed, expected, log2_enrichment, residual, q_values, long

arm_results = {}
all_long = []
for arm in arm_order:
    arm_results[arm] = build_arm_enrichment_table(syllable_probs, arm)
    all_long.append(arm_results[arm][-1])

enrichment_results = pd.concat(all_long, ignore_index=True)
enrichment_results.to_csv(OUT_DIR / 'syllable_species_enrichment_by_arm.csv', index=False)
display(enrichment_results.sort_values('q_value').head(20))

## Species-by-Syllable Heatmaps

In [ ]:
top_syllables = (
    syllable_probs.groupby('moseq_cluster')['soft_frames']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_SYLLABLES)
    .index.astype(int)
    .tolist()
)

bound = np.nanpercentile(
    np.abs(enrichment_results.loc[enrichment_results['moseq_cluster'].isin(top_syllables), 'log2_enrichment_vs_arm_expected']),
    98,
)
bound = max(float(bound), 1.0)

fig, axes = plt.subplots(len(arm_order), 1, figsize=(max(12, 0.52 * len(top_syllables)), max(3.4 * len(arm_order), 4)), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, arm in zip(axes, arm_order):
    _, _, log2_enrichment, _, q_values, _ = arm_results[arm]
    heat = log2_enrichment.reindex(index=species_order, columns=top_syllables, fill_value=0.0)
    sig = q_values.reindex(index=species_order, columns=top_syllables, fill_value=np.nan) < 0.05
    image = ax.imshow(heat.to_numpy(float), aspect='auto', cmap='coolwarm', vmin=-bound, vmax=bound)
    for y, x in np.argwhere(sig.to_numpy(bool)):
        ax.text(x, y, '*', ha='center', va='center', color='black', fontsize=7)
    ax.set_title(f'Aligned arm {arm}: syllable enrichment by species')
    ax.set_yticks(np.arange(len(species_order)))
    ax.set_yticklabels([s.replace('_', ' ') for s in species_order], fontsize=8)
    ax.set_xticks(np.arange(len(top_syllables)))
    ax.set_xticklabels([wrapped_syllable_label(sid) for sid in top_syllables], rotation=55, ha='right', fontsize=7)
    ax.set_ylabel('species')
    fig.colorbar(image, ax=ax, shrink=0.85, label='log2 observed / expected')
axes[-1].set_xlabel('MoSeq syllable')
plt.show()

## Pairwise Species Comparisons

In [ ]:
pairwise_rows = []
for arm in arm_order:
    observed, _, _, _, _, _ = arm_results[arm]
    probs = observed.div(observed.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    distance = pd.DataFrame(
        pairwise_distances(probs.to_numpy(float), metric='cosine'),
        index=probs.index,
        columns=probs.index,
    )
    distance.to_csv(OUT_DIR / f'aligned_arm_{arm}_species_syllable_cosine_distance.csv')
    for i, s1 in enumerate(distance.index):
        for s2 in distance.columns[i + 1:]:
            pairwise_rows.append({
                'aligned_arm': arm,
                'species_1': s1,
                'species_2': s2,
                'cosine_distance': float(distance.loc[s1, s2]),
            })

    if len(probs) > 2:
        condensed = squareform(distance.to_numpy(float), checks=False)
        order = leaves_list(linkage(condensed, method='average'))
    else:
        order = np.arange(len(probs))
    ordered = distance.iloc[order, order]
    plt.figure(figsize=(5.8, 5.0))
    sns.heatmap(ordered, cmap='mako', square=True, vmin=0, annot=True, fmt='.2f', cbar_kws={'label': 'cosine distance'})
    plt.title(f'Aligned arm {arm}: species distances from syllable profiles')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

pairwise_species_distances = pd.DataFrame(pairwise_rows)
pairwise_species_distances.to_csv(OUT_DIR / 'pairwise_species_syllable_distances_by_arm.csv', index=False)
display(pairwise_species_distances.sort_values('cosine_distance', ascending=False).head(20))

## Top Enriched Syllables Per Species

In [ ]:
TOP_PER_SPECIES = 8

top_enriched = (
    enrichment_results
    .query('soft_frames >= @MIN_SOFT_FRAMES')
    .sort_values(['aligned_arm', 'species', 'log2_enrichment_vs_arm_expected'], ascending=[True, True, False])
    .groupby(['aligned_arm', 'species'], as_index=False)
    .head(TOP_PER_SPECIES)
    .loc[:, ['aligned_arm', 'species', 'moseq_cluster', 'moseq_label', 'soft_frames', 'expected_soft_frames', 'log2_enrichment_vs_arm_expected', 'standardized_residual', 'q_value']]
)
top_enriched.to_csv(OUT_DIR / 'top_enriched_syllables_per_species_by_arm.csv', index=False)
display(top_enriched)

for arm in arm_order:
    plot_df = top_enriched.loc[top_enriched['aligned_arm'] == arm].copy()
    plot_df['label'] = plot_df['moseq_cluster'].astype(str) + ': ' + plot_df['moseq_label'].astype(str).str.replace('_', ' ')
    g = sns.catplot(
        data=plot_df,
        y='label',
        x='log2_enrichment_vs_arm_expected',
        col='species',
        col_wrap=4,
        kind='bar',
        sharey=False,
        height=3.2,
        aspect=1.05,
        color='0.35',
    )
    g.set_axis_labels('log2 observed / expected', '')
    g.set_titles('{col_name}')
    g.fig.suptitle(f'Aligned arm {arm}: top enriched syllables per species', y=1.02)
    plt.show()